In [ ]:
from dotenv import load_dotenv

from langchain_teddynote import logging
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
import faiss

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-09")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)

In [ ]:
loader1 = TextLoader("data/nlp-keywords.txt")
loader2 = TextLoader("data/finance-keywords.txt")

In [ ]:
split_doc1 = loader1.load_and_split(text_splitter)
split_doc2 = loader2.load_and_split(text_splitter)

len(split_doc1), len(split_doc2)

# VectorStore

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
dimension_size = len(embeddings.embed_query("hello world"))  # 임베딩 차원 크기

print(dimension_size)

In [ ]:
db = FAISS(
    embedding_function=embeddings, 
    index=faiss.INdexFlastL2(dimension_size), 
    docstore=InMemoryDocstore(), 
    index_to_docstore_id={}
)

벡터 저장소 생성 (from_documents)

In [ ]:
db1 = FAISS.from_documents(documents=split_doc1, embedding=OpenAIEmbeddings())

In [ ]:
db1.index_to_docstore_id  # 문서 저장소 id 확인

In [ ]:
db1.docstore._dict  # 저장된 문서의 ID: Document 확인

벡터 저장소 생성 (from_texts)

In [ ]:
db2 = FAISS.from_texts(
    ["안녕하세요. 정말 반갑습니다.", "제 이름은 테디입니다."],
    embedding=OpenAIEmbeddings(),
    metadatas=[{"source": "텍스트문서"}, {"source": "텍스트문서"}],
    ids=["doc1", "doc2"],
)

In [ ]:
db2.docstore._dict

유사도 검색

In [ ]:
db1.similarity_search("TF IDF 에 대하여 알려줘")

In [ ]:
db1.similarity_search("TF IDF 에 대하여 알려줘", k=2)

In [ ]:
db1.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "data/nlp-keywords.txt"}, k=2
)

In [ ]:
db1.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "data/finance-keywords.txt"}, k=2
)

문서로부터 추가

In [ ]:
db1.add_documents(
    [
        Document(
            page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요",
            metadata={"source": "mydata.txt"}
        )
    ], 
    ids=["new_doc1"]
)

In [ ]:
db.similarity_search("안녕하세요", k=1)

텍스트로부터 추가

In [ ]:
db1.add_texts(
    ["이번엔 텍스트 데이터를 추가합니다.", "추가한 2번째 텍스트 데이터 입니다."],
    metadatas=[{"source": "mydata.txt"}, {"source": "mydata.txt"}],
    ids=["new_doc2", "new_doc3"]
)

In [ ]:
db.index_to_docstore_id

문서 삭제

In [ ]:
# 삭제용 데이터 추가
ids = db1.add_texts(
    ["삭제용 데이터를 추가합니다.", "2번째 삭제용 데이터입니다."],
    metadatas=[{"source": "mydata.txt"}, {"source": "mydata.txt"}],
    ids=["delete_doc1", "delete_doc2"],
)

print(ids)  # 삭제할 데이터의 id

In [ ]:
db1.delete(ids)  # 삭제하기기

In [ ]:
db1.index_to_docstore_id  # 삭제된 결과 출력

## 저장 및 불러오기

로컬에 저장

In [ ]:
db1.save_local(folder_path="faiss_db", index_name="faiss_index")

로컬에서 불러오기

In [ ]:
loaded_db1 = FAISS.load_local(
    folder_path="faiss_db", 
    index_name="faiss_index", 
    embeddings=embeddings, 
    allow_dangerous_deserialization=True
)

In [ ]:
loaded_db1.index_to_docstore_id  # 로드된 데이터 확인

FAISS 객체 병합

In [ ]:
db4merge1 = FAISS.load_local(
    folder_path="faiss_db",
    index_name="faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

In [ ]:
# 불러온 데이터와 병합할 새로운 FAISS 벡터DB 생성
db4merge2 = FAISS.from_documents(documents=split_doc2, embedding=OpenAIEmbedding())

In [ ]:
db4merge1.index_to_docstore_id

In [ ]:
db4merge2.index_to_docstore_id

In [ ]:
# 데이터 병합하기
db4merge1.merge_from(db4merge2)

In [ ]:
db4merge1.index_to_docstore_id

## Retriever

In [ ]:
db4retriever = FAISS.from_documents(
    documents=split_doc1 + split_doc2, embedding=OpenAIEmbeddings()
)

In [ ]:
# 검색기로 변환
retriever = db4retriever.as_retriever()  # 기본적으로 문서 4개 반환

In [ ]:
# 검색 수행
retriever.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# MMR 검색
retriever_mmr = db.as_retriever(
    search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25, "fetch_k": 10}
)

In [ ]:
retriever_mmr.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 임계값 기반 검색
retriever_thres = db.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.8}
)

In [ ]:
retriever_thres.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 가장 유사한 문서 하나만 검색
retriver_one = db.as_retriever(search_kwargs={"k": 1})

In [ ]:
retriver_one.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 특정 메타데이터 필터 적용
retriever_meta = db.as_retriever(
    search_kwargs={"filter": {"source": "data/finance-keywords.txt"}, "k": 2}
)

In [ ]:
retriever_meta.invoke("ESG 에 대하여 알려줘")